# Agent Accessibility Benchmark — pre-registered analysis (B11.3′)

This notebook only **calls** `eval/agentbench/analysis.py`; every statistic is defined there and in
`docs/agentbench/preregistration.md`, both committed and tagged before any Week 12 run.
Do not add an analysis here that the pre-registration does not list — report it as exploratory, separately.

Input: `EpisodeRecord` JSON lines written by the runner (A12.1′).

In [ ]:
import sys, pathlib
REPO = pathlib.Path.cwd()
while not (REPO / 'pyproject.toml').exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from eval.agentbench import analysis as A

RESULTS = REPO / 'runs' / 'v1' / 'episodes.jsonl'   # set to the run being analysed
records = A.load_records(RESULTS)
len(records)

## 0. Exclusions (reported before any result)

In [ ]:
from collections import Counter
Counter(r.excluded_reason or ('evaluator indeterminate' if r.outcome == 'indeterminate' else 'usable') for r in records)

## 1. H1 — name defects reduce tree-only agent success (primary)
Exact McNemar per (defect × model), Holm across the H1 family; paired difference with Bonett–Price interval; task-cluster bootstrap as sensitivity.

In [ ]:
H1_DEFECTS = ['strip_accessible_name', 'click_only_widget', 'detach_label']   # as pre-registered
models = sorted({r.model for r in records if r.agent == 'tree'})
h1 = {}
for model in models:
    for defect in H1_DEFECTS:
        pairs, dropped = A.pair_episodes(records, defect=defect, agent='tree', model=model)
        if pairs:
            h1[f'{model} / {defect}'] = A.paired_effect(pairs, dropped)
adjusted = A.holm({k: v['mcnemar_p'] for k, v in h1.items()})
for k, v in h1.items():
    print(f"{k:45} n={v['pairs']:4} diff={v['diff']:+.3f} CI={v['diff_ci'][0]:+.3f}..{v['diff_ci'][1]:+.3f} "
          f"p={v['mcnemar_p']:.4f} p_holm={adjusted[k]:.4f}")

## 2. H2 — vision-capable agents are less affected (difference in differences)

In [ ]:
for model in models:
    for defect in H1_DEFECTS:
        tree, _ = A.pair_episodes(records, defect=defect, agent='tree', model=model)
        vision, _ = A.pair_episodes(records, defect=defect, agent='browser_use', model=model)
        if tree and vision:
            print(model, defect, A.difference_in_differences(tree, vision))

## 3. H3 — keyboard and focus defects hurt the screen-reader agent most

In [ ]:
H3_DEFECTS = ['keyboard_trap', 'remove_from_tab_order', 'outline_none']   # as pre-registered
for model in models:
    for defect in H3_DEFECTS:
        sr, _ = A.pair_episodes(records, defect=defect, agent='screen_reader', model=model)
        tree, _ = A.pair_episodes(records, defect=defect, agent='tree', model=model)
        if sr and tree:
            print(model, defect, A.difference_in_differences(sr, tree))

## 4. H4 — accessible pages cost fewer tokens per successful task

In [ ]:
cells = {}
for r in records:
    cells.setdefault((r.agent, r.model, r.defect), []).append(r)
for (agent, model, defect), rs in sorted(cells.items()):
    print(f'{agent:14} {model:22} {defect:24} tokens/success = {A.tokens_per_success(rs)}')